# Video Clone — Colab remote worker

Local desktop app remains the **control plane** (projects, jobs, SQLite).
This notebook starts a thin FastAPI **inference worker** and exposes it via Cloudflare Tunnel.

1. Run install cell
2. Set `SHARED_SECRET`
3. Start API + tunnel
4. Paste URL + secret into app **Cấu hình → Remote**

In [ ]:
!pip -q install fastapi uvicorn faster-whisper edge-tts httpx
# Optional: cloudflared binary — install per Colab instructions

In [ ]:
import secrets
SHARED_SECRET = secrets.token_urlsafe(16)
print('SHARED_SECRET =', SHARED_SECRET)
print('Put this in the desktop app Settings → remote_secret')

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI(title='VideoClone Colab Worker')

def check(secret: str | None):
    if secret != SHARED_SECRET:
        raise HTTPException(401, 'bad secret')

@app.get('/health')
def health(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'ok': True, 'worker': 'colab'}

@app.get('/capabilities')
def caps(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'asr': True, 'tts': True}

class AsrReq(BaseModel):
    # Desktop may upload/pull audio in a later iteration
    note: str = 'stub — wire faster-whisper here'

@app.post('/infer/asr')
def infer_asr(body: AsrReq, x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'status': 'not_implemented', 'hint': 'Connect chunked audio upload from local control plane'}

def run():
    uvicorn.run(app, host='127.0.0.1', port=8765)

threading.Thread(target=run, daemon=True).start()
print('Worker on http://127.0.0.1:8765')

## Tunnel

Install and run cloudflared, then copy the `https://*.trycloudflare.com` URL into the desktop app.

```bash
cloudflared tunnel --url http://127.0.0.1:8765
```

Desktop **readiness** will probe `GET {remote_url}/health` with header `X-VC-Secret`.